# Mushroom Classification: Edible vs Poisonous

Goal: predict if a mushroom is edible (`e`) or poisonous (`p`) from its features.
Data: Kaggle competition dataset.

**Steps:** Look at data -> Clean data -> Build pipeline -> Compare models -> Tune best model -> Make submission

**Result:** All models get about 100% accuracy in 5-fold cross-validation. Final model: tuned LightGBM.

> Note: this notebook was run on Kaggle, so file paths start with `/kaggle/input/...`. To run it on your computer, download the CSV files and change the paths in step 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
path = '/kaggle/input/competitions/may-2026kaggle-assignment-3/'
train = pd.read_csv(path + 'train.csv')
test = pd.read_csv(path + 'test.csv')
sample_submission = pd.read_csv(path + 'sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
train.head()

## 2. Quick Look at the Data

In [ ]:
# Number columns and text (category) columns
num_cols_all = train.select_dtypes(include=[np.number]).columns.tolist()
print('Number columns:', num_cols_all)

# Basic stats for the two real number features
num_feature_cols = ['number_of_bruises', 'ring-number']
train[num_feature_cols].describe().T

## 3. Clean the Data

**Missing values:** Missing `odor` is linked with the class, so we do not delete it. We fill missing text values with the word `"missing"`, so the model can use it as a category.

In [ ]:
# How many missing values in each column?
missing = train.isnull().sum()
missing[missing > 0]

In [ ]:
# Is a missing odor linked to the class?
pd.crosstab(train['odor'].isna(), train['class'])

In [ ]:
# Fill missing text values with 'missing'
text_cols = [c for c in train.columns
             if c not in ['ID', 'mushroom_id', 'class'] and c not in num_feature_cols]

for c in text_cols:
    train[c] = train[c].fillna('missing').astype(str)
    test[c] = test[c].fillna('missing').astype(str)

In [ ]:
# Remove duplicate rows
check_cols = [c for c in train.columns if c not in ['ID', 'mushroom_id']]
n_dupes = train.duplicated(subset=check_cols).sum()
print('Duplicate rows:', n_dupes)

train = train.drop_duplicates(subset=check_cols).reset_index(drop=True)
print('Train shape after cleaning:', train.shape)

**Outliers:** Bruise counts are 0 to 24 and ring counts are 1 to 2. These are real values, so no rows are removed.

## 4. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class balance
sns.countplot(data=train, x='class', order=['e', 'p'], ax=axes[0])
axes[0].set_title('Class distribution')

# Odor vs class
pd.crosstab(train['odor'], train['class']).plot(
    kind='bar', stacked=True, ax=axes[1], color=['#4C9F70', '#C0392B'])
axes[1].set_title('Odor vs Class')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Habitat values in train and test
print('Train habitat:', sorted(train['habitat'].unique()))
print('Test habitat :', sorted(test['habitat'].unique()))

`habitat` has no common values in train and test, so the model cannot learn from it. We drop it.

## 5. Preprocessing Pipeline

- Drop `habitat` (no common values in train and test) and `veil-type` (same value in every row).
- Text features: `OneHotEncoder`. Unknown values in test become all zeros.
- Number features: fill missing with the median, then scale with `StandardScaler`.

In [ ]:
drop_cols = ['ID', 'mushroom_id', 'class', 'habitat', 'veil-type']
feature_cols = [c for c in train.columns if c not in drop_cols]

cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]

X = train[feature_cols]
y = train['class'].map({'e': 0, 'p': 1})
X_test = test[feature_cols]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num_cols)
])

## 6. Compare Models

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, verbosity=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = scores.mean()
    print(f'{name:20s} CV accuracy = {scores.mean():.5f}')

In [ ]:
# Bar chart of results
results_s = pd.Series(results).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=results_s.values, y=results_s.index)
plt.xlabel('Mean CV Accuracy')
plt.title('Model Comparison')
plt.xlim(0.9, 1.01)
plt.tight_layout()
plt.show()

## 7. Tune the Best Model (LightGBM)

In [ ]:
param_grid = {
    'model__n_estimators': [200, 400],
    'model__num_leaves': [15, 31],
    'model__learning_rate': [0.05, 0.1],
}

lgbm_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMClassifier(random_state=RANDOM_STATE, verbosity=-1))
])

search = GridSearchCV(lgbm_pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
search.fit(X, y)

print('Best parameters:', search.best_params_)
print('Best CV accuracy:', search.best_score_)

## 8. Final Model

In [ ]:
# The best pipeline is already trained on all training data
final_model = search.best_estimator_

preds = final_model.predict(X)
print('Training accuracy:', accuracy_score(y, preds))
print(classification_report(y, preds, target_names=['edible', 'poisonous']))

sns.heatmap(confusion_matrix(y, preds), annot=True, fmt='d', cmap='Blues',
            xticklabels=['edible', 'poisonous'], yticklabels=['edible', 'poisonous'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 9. Make Submission

In [ ]:
test_preds = final_model.predict(X_test)

submission = pd.DataFrame({
    'ID': test['ID'],
    'class': pd.Series(test_preds).map({0: 'e', 1: 'p'})
})

# Quick checks
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(sample_submission)

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv:', submission.shape)
submission['class'].value_counts()